In [1]:
import joblib
import pandas as pd
import numpy as np
import os

# Check files exist and aren't corrupted/empty
for fname in ['xgb_wait_time_model.pkl', 'xgb_demand_model.pkl']:
    size = os.path.getsize(fname)
    print(f"{fname}: {size / 1024:.1f} KB")
    assert size > 1000, f"File too small — probably corrupt: {fname}"

# Load both
model_wait   = joblib.load('xgb_wait_time_model.pkl')
model_demand = joblib.load('xgb_demand_model.pkl')

print("\nWait model type  :", type(model_wait))
print("Demand model type:", type(model_demand))
print("Wait model trees :", model_wait.n_estimators)
print("Demand model trees:", model_demand.n_estimators)

xgb_wait_time_model.pkl: 837.4 KB
xgb_demand_model.pkl: 846.7 KB

Wait model type  : <class 'xgboost.sklearn.XGBRegressor'>
Demand model type: <class 'xgboost.sklearn.XGBRegressor'>
Wait model trees : 200
Demand model trees: 200


In [2]:
feature_cols = [
    'pickup_zone', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'is_weekend', 'demand_lag_1h', 'demand_lag_2h', 'demand_lag_24h',
    'demand_rolling_3h', 'demand_rolling_24h', 'avg_distance'
]

# One fake row — evening rush, weekday, moderate lag
test_row = pd.DataFrame([{
    'pickup_zone'        : 23,
    'hour_sin'           : np.sin(2 * np.pi * 17 / 24),   # 5 PM
    'hour_cos'           : np.cos(2 * np.pi * 17 / 24),
    'day_sin'            : np.sin(2 * np.pi * 4 / 7),     # Friday
    'day_cos'            : np.cos(2 * np.pi * 4 / 7),
    'is_weekend'         : 0,
    'demand_lag_1h'      : 15,
    'demand_lag_2h'      : 12,
    'demand_lag_24h'     : 18,
    'demand_rolling_3h'  : 14.0,
    'demand_rolling_24h' : 11.5,
    'avg_distance'       : 7.2
}])[feature_cols]

pred_wait   = model_wait.predict(test_row)[0]
pred_demand = model_demand.predict(test_row)[0]

print(f"Predicted wait time : {pred_wait:.2f} minutes")
print(f"Predicted demand    : {pred_demand:.2f} rides/hr")

# Basic sanity bounds
assert 0 < pred_wait < 120,  f"Wait prediction out of range: {pred_wait}"
assert 0 < pred_demand < 200, f"Demand prediction out of range: {pred_demand}"
print("\nSingle row test PASSED")

Predicted wait time : 43.56 minutes
Predicted demand    : 1.17 rides/hr

Single row test PASSED


In [3]:
# Reproduce the actual predict_zone_rankings function
def test_all_zones(model_wait, model_demand, feature_cols, n_zones=50):
    hour = 17
    dow  = 4  # Friday

    rows = []
    for zone in range(1, n_zones + 1):
        rows.append({
            'pickup_zone'        : zone,
            'hour_sin'           : np.sin(2 * np.pi * hour / 24),
            'hour_cos'           : np.cos(2 * np.pi * hour / 24),
            'day_sin'            : np.sin(2 * np.pi * dow / 7),
            'day_cos'            : np.cos(2 * np.pi * dow / 7),
            'is_weekend'         : 0,
            'demand_lag_1h'      : 10,
            'demand_lag_2h'      : 8,
            'demand_lag_24h'     : 12,
            'demand_rolling_3h'  : 10.0,
            'demand_rolling_24h' : 9.0,
            'avg_distance'       : 7.2
        })

    pred_df = pd.DataFrame(rows)[feature_cols]

    wait_preds   = model_wait.predict(pred_df)
    demand_preds = model_demand.predict(pred_df)

    results = pd.DataFrame({
        'zone'             : range(1, n_zones + 1),
        'predicted_wait'   : wait_preds.round(2),
        'predicted_demand' : demand_preds.round(2),
        'score'            : (demand_preds / (wait_preds + 1)).round(3)
    }).sort_values('score', ascending=False).reset_index(drop=True)

    # Checks
    assert results['predicted_wait'].isna().sum()   == 0, "NaNs in wait predictions"
    assert results['predicted_demand'].isna().sum() == 0, "NaNs in demand predictions"
    assert (results['predicted_wait'] > 0).all(),   "Negative wait time predicted"
    assert (results['predicted_demand'] > 0).all(), "Negative demand predicted"

    print("Top 5 zones:")
    print(results.head(5).to_string(index=False))
    print("\nAll zones test PASSED")
    return results

results = test_all_zones(model_wait, model_demand, feature_cols)

Top 5 zones:
 zone  predicted_wait  predicted_demand  score
   45       41.610001              1.39  0.033
   44       41.680000              1.39  0.033
   43       42.150002              1.39  0.032
   47       43.330002              1.36  0.031
   40       42.369999              1.35  0.031

All zones test PASSED


In [4]:
print("\n--- Edge Case Tests ---")

# 1. Midnight (hour = 0)
midnight_row = test_row.copy()
midnight_row['hour_sin'] = np.sin(0)
midnight_row['hour_cos'] = np.cos(0)
p = model_wait.predict(midnight_row)[0]
assert not np.isnan(p), "NaN at midnight"
print(f"Midnight prediction   : {p:.2f} min  PASS")

# 2. Weekend
weekend_row = test_row.copy()
weekend_row['is_weekend'] = 1
weekend_row['day_sin'] = np.sin(2 * np.pi * 6 / 7)  # Sunday
weekend_row['day_cos'] = np.cos(2 * np.pi * 6 / 7)
p = model_wait.predict(weekend_row)[0]
assert not np.isnan(p), "NaN on weekend"
print(f"Weekend prediction    : {p:.2f} min  PASS")

# 3. Zero lag (new zone with no history)
zero_lag_row = test_row.copy()
for col in ['demand_lag_1h','demand_lag_2h','demand_lag_24h','demand_rolling_3h','demand_rolling_24h']:
    zero_lag_row[col] = 0
p = model_wait.predict(zero_lag_row)[0]
assert not np.isnan(p), "NaN with zero lags"
print(f"Zero lag prediction   : {p:.2f} min  PASS")

# 4. Very high demand lag (peak scenario)
peak_row = test_row.copy()
peak_row['demand_lag_1h']     = 100
peak_row['demand_lag_24h']    = 90
peak_row['demand_rolling_3h'] = 95.0
p = model_wait.predict(peak_row)[0]
assert not np.isnan(p), "NaN at peak demand"
print(f"Peak demand prediction: {p:.2f} min  PASS")

print("\nAll edge cases PASSED")


--- Edge Case Tests ---
Midnight prediction   : 45.01 min  PASS
Weekend prediction    : 43.40 min  PASS
Zero lag prediction   : 45.08 min  PASS
Peak demand prediction: 43.56 min  PASS

All edge cases PASSED


In [5]:
# XGBoost stores the exact feature names it was trained on
trained_features = model_wait.get_booster().feature_names
your_features    = feature_cols

mismatches = set(trained_features) ^ set(your_features)

if mismatches:
    print("MISMATCH — features differ:", mismatches)
else:
    print("Feature names match exactly  PASS")

# Also check order
if list(trained_features) != your_features:
    print("WARNING — feature ORDER differs. This can silently corrupt predictions.")
    print("Model trained order:", trained_features)
    print("Your order         :", your_features)
else:
    print("Feature order matches  PASS")

Feature names match exactly  PASS
Feature order matches  PASS
